In [1]:
data=open('训练文本.flare').read()
#移除换行符
data=data.replace('\n',' ').replace('\r',' ')
print(data)

flare is a teacher in ai industry. He obtained his phd inAustralia. Australia is a country in the Southern Hemissphere flare is a teacher in ai industry. He obtained his phd inAustralia. Australia is a country in the Southern Hemissphere flare is a teacher in ai industry. He obtained his phd inAustralia. Australia is a country in the Southern Hemissphere flare is a teacher in ai industry. He obtained his phd inAustralia. Australia is a country in the Southern Hemissphere flare is a teacher in ai industry. He obtained his phd inAustralia. Australia is a country in the Southern Hemissphere flare is a teacher in ai industry. He obtained his phd inAustralia. Australia is a country in the Southern Hemissphere flare is a teacher in ai industry. He obtained his phd inAustralia. Australia is a country in the Southern Hemissphere flare is a teacher in ai industry. He obtained his phd inAustralia. Australia is a country in the Southern Hemissphere flare is a teacher in ai industry. He obtained h

In [2]:
#去重
letters=list(set(data))
print(letters)
num_letters=len(letters)
print(num_letters)

['p', ' ', 'r', 'h', 'c', 'y', '.', 'H', 'd', 'l', 'a', 'A', 'f', 's', 'u', 'm', 'i', 'b', 'S', 't', 'e', 'o', 'n']
23


In [3]:
#字典
int_to_char={a:b for a,b in enumerate(letters)}
char_to_int={b:a for a,b in enumerate(letters)}
print(int_to_char)
print(char_to_int)

{0: 'p', 1: ' ', 2: 'r', 3: 'h', 4: 'c', 5: 'y', 6: '.', 7: 'H', 8: 'd', 9: 'l', 10: 'a', 11: 'A', 12: 'f', 13: 's', 14: 'u', 15: 'm', 16: 'i', 17: 'b', 18: 'S', 19: 't', 20: 'e', 21: 'o', 22: 'n'}
{'p': 0, ' ': 1, 'r': 2, 'h': 3, 'c': 4, 'y': 5, '.': 6, 'H': 7, 'd': 8, 'l': 9, 'a': 10, 'A': 11, 'f': 12, 's': 13, 'u': 14, 'm': 15, 'i': 16, 'b': 17, 'S': 18, 't': 19, 'e': 20, 'o': 21, 'n': 22}


In [4]:
time_step=20

In [5]:
import numpy as np
from tensorflow.keras.utils import to_categorical
#滑动窗口提取数据
def extract_data(data, slide):
    x=[]
    y=[]
    for i in range(len(data) - slide):
        x.append([a for a in data[i:i+slide]])
        y.append(data[i+slide])
    return x,y
#字符到数字的批量转化
def char_to_int_Data(x,y, char_to_int):
    x_to_int=[]
    y_to_int=[]
    for i in range(len(x)):
        x_to_int.append([char_to_int[char] for char in x[i]])
        y_to_int.append([char_to_int[char] for char in y[i]])
    return x_to_int, y_to_int
#实现输入字符文章的批量处理,输入整个字符、滑动窗口大小、转化字典
def data_preprocessing(data, slide, num_letters, char_to_int):
    char_Data = extract_data(data, slide)
    int_Data = char_to_int_Data(char_Data[0], char_Data[1], char_to_int)
    Input=int_Data[0]
    Output=list(np.array(int_Data[1]).flatten())
    Input_RESHAPED = np.array(Input).reshape(len(Input), slide) 
    new = np.random.randint(0,10,size=[Input_RESHAPED.shape[0],Input_RESHAPED.shape[1],num_letters])
    for i in range(Input_RESHAPED.shape[0]):
        for j in range(Input_RESHAPED.shape[1]):
            new[i,j,:] = to_categorical(Input_RESHAPED[i,j],num_classes=num_letters)
    return new, Output

2025-11-03 20:09:26.010873: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-03 20:09:26.010938: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-03 20:09:26.012087: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [6]:
X,y=data_preprocessing(data,time_step,num_letters,char_to_int)

In [7]:
print(X.shape,len(y))

(47461, 20, 23) 47461


In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.1,random_state=10)
print(X_train.shape,len(y_train))

(42714, 20, 23) 42714


In [9]:
y_train_category=to_categorical(y_train,num_letters)
print(y_train_category)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [21]:
from keras.models import Sequential
from keras.layers import Dense,LSTM,Input

model=Sequential()
model.add(Input(shape=(X_train.shape[1],X_train.shape[2])))
model.add(LSTM(units=20,activation='relu'))
model.add(Dense(units=num_letters,activation='softmax'))
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_2 (LSTM)               (None, 20)                3520      
                                                                 
 dense_2 (Dense)             (None, 23)                483       
                                                                 
Total params: 4003 (15.64 KB)
Trainable params: 4003 (15.64 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [43]:
model.fit(X_train,y_train_category,batch_size=1000,epochs=5)

Epoch 1/5
43/43 [==============================] - 2s 34ms/step - loss: 0.1030 - accuracy: 0.9801
Epoch 2/5
43/43 [==============================] - 1s 24ms/step - loss: 0.0686 - accuracy: 0.9872
Epoch 3/5
43/43 [==============================] - 1s 32ms/step - loss: 0.0460 - accuracy: 0.9983
Epoch 4/5
43/43 [==============================] - 1s 32ms/step - loss: 0.0311 - accuracy: 1.0000
Epoch 5/5
43/43 [==============================] - 1s 29ms/step - loss: 0.0213 - accuracy: 1.0000


In [44]:
y_train_predict=np.argmax(model.predict(X_train), axis=-1)
print(y_train_predict)

1335/1335 [==============================] - 6s 4ms/step
[14 16  7 ... 16  1 10]


In [45]:
y_train_predict_char = [int_to_char[i] for i in y_train_predict]
print(y_train_predict_char)

['u', 'i', 'H', 'd', 'h', 'p', 'u', 'o', ' ', 'l', 'a', 's', 't', 'n', 'l', ' ', ' ', 'l', 'i', 'd', 'n', 'a', 'l', 'c', 'e', 'u', 't', 'e', ' ', 'd', 'e', 'p', 'u', 'e', ' ', 'H', 'e', 'a', ' ', 'y', 'e', 'u', 'i', 'e', 'u', 'y', 'a', 'e', 'r', 'y', 'i', 'r', 't', 'i', 'u', 'e', 'r', 'i', 'i', 't', 't', '.', 's', ' ', 'a', 'A', 'A', 'i', ' ', 'a', 'i', 'h', ' ', 'u', 't', 'd', 's', 'h', 'a', 'r', 'a', 's', 'a', 'd', 'e', 'a', 'a', ' ', 'i', ' ', 'l', 'n', '.', 't', 'i', 'e', 'r', 'i', 'e', 'd', 'i', 'y', 'o', 'e', 'l', 'r', 'i', 'a', 'a', 'a', 't', 'a', 'a', 's', 'H', 'r', 't', ' ', 'e', ' ', 't', 's', 'i', 'r', ' ', ' ', 's', '.', 'o', ' ', 'r', 'a', ' ', ' ', 'e', 'o', 'm', 'a', ' ', 'i', 'a', 's', 'e', 't', 'e', ' ', 't', 's', 'm', ' ', ' ', 'e', 'r', ' ', 'e', 'a', 'i', 'a', '.', 't', 'h', ' ', ' ', 'e', 'n', 'h', 'a', 'h', ' ', 'r', 'i', 'u', 'n', 'a', 'i', 'h', ' ', ' ', 's', 'i', 't', 't', 'p', 'h', ' ', 'r', ' ', 't', 'A', 'r', 's', 'e', ' ', 't', 'r', 'p', 'a', 'f', 'i', 'e',

In [46]:
from sklearn.metrics import accuracy_score
accuracy_train=accuracy_score(y_train,y_train_predict)
print(accuracy_train)

1.0


In [47]:
y_test_predict=np.argmax(model.predict(X_test), axis=-1)
accuracy_test=accuracy_score(y_test,y_test_predict)
print(accuracy_test)
print(y_test_predict)
print(y_test)

149/149 [==============================] - 1s 5ms/step
1.0
[ 2  1 14 ...  5 16 19]
[2, 1, 14, 0, 1, 1, 20, 20, 18, 3, 4, 2, 1, 14, 22, 13, 10, 13, 19, 22, 19, 10, 13, 21, 1, 1, 13, 10, 19, 20, 7, 2, 20, 3, 16, 0, 21, 9, 4, 16, 13, 16, 10, 20, 1, 9, 20, 0, 1, 20, 7, 3, 1, 8, 1, 19, 13, 20, 10, 2, 5, 7, 1, 3, 13, 19, 20, 1, 13, 21, 21, 20, 7, 22, 1, 1, 9, 10, 20, 9, 4, 3, 2, 10, 7, 16, 13, 13, 8, 5, 10, 1, 22, 16, 1, 3, 14, 20, 1, 2, 18, 2, 1, 4, 22, 2, 20, 16, 1, 0, 20, 7, 1, 4, 3, 12, 3, 10, 4, 1, 9, 13, 15, 21, 20, 2, 13, 1, 7, 16, 16, 22, 2, 7, 20, 20, 20, 13, 20, 14, 3, 20, 20, 16, 1, 12, 1, 16, 9, 17, 16, 20, 2, 1, 14, 10, 20, 22, 19, 16, 14, 15, 3, 12, 10, 20, 16, 14, 19, 22, 1, 16, 1, 1, 2, 14, 13, 1, 10, 1, 2, 13, 8, 19, 1, 16, 2, 3, 4, 14, 19, 20, 10, 10, 20, 9, 1, 12, 16, 10, 2, 0, 11, 10, 8, 22, 16, 6, 21, 16, 21, 20, 13, 14, 6, 19, 2, 10, 10, 14, 22, 19, 8, 13, 7, 12, 22, 22, 16, 3, 20, 22, 20, 10, 9, 13, 13, 1, 22, 19, 17, 10, 20, 1, 19, 13, 20, 20, 5, 14, 14, 1, 22, 16, 16

In [48]:
new_letters='flare is a teacher in ai industry. He obtained his phd in Australia.'
X_new,y_new=data_preprocessing(new_letters,time_step,num_letters,char_to_int)
y_new_predict=np.argmax(model.predict(X_new), axis=-1)
print(y_new_predict)

2/2 [==============================] - 0s 7ms/step
[22  1 10 16  1 16 22  8 14 13 19  2  5  6  1  7 20  1 21 17 19 10 16 22
 20  8  1  3 16 13  1  0  3  8  1 16 22 11 19 14 17 19  2 20  9 16 10  6]


In [49]:
y_new_predict_char = [int_to_char[i] for i in y_new_predict]
print(y_new_predict_char)

['n', ' ', 'a', 'i', ' ', 'i', 'n', 'd', 'u', 's', 't', 'r', 'y', '.', ' ', 'H', 'e', ' ', 'o', 'b', 't', 'a', 'i', 'n', 'e', 'd', ' ', 'h', 'i', 's', ' ', 'p', 'h', 'd', ' ', 'i', 'n', 'A', 't', 'u', 'b', 't', 'r', 'e', 'l', 'i', 'a', '.']


In [50]:
for i in range(0,X_new.shape[0]-time_step):
    print(new_letters[i:i+20],'--predict next letter is--',y_new_predict_char[i])

flare is a teacher i --predict next letter is-- n
lare is a teacher in --predict next letter is--  
are is a teacher in  --predict next letter is-- a
re is a teacher in a --predict next letter is-- i
e is a teacher in ai --predict next letter is--  
 is a teacher in ai  --predict next letter is-- i
is a teacher in ai i --predict next letter is-- n
s a teacher in ai in --predict next letter is-- d
 a teacher in ai ind --predict next letter is-- u
a teacher in ai indu --predict next letter is-- s
 teacher in ai indus --predict next letter is-- t
teacher in ai indust --predict next letter is-- r
eacher in ai industr --predict next letter is-- y
acher in ai industry --predict next letter is-- .
cher in ai industry. --predict next letter is--  
her in ai industry.  --predict next letter is-- H
er in ai industry. H --predict next letter is-- e
r in ai industry. He --predict next letter is--  
 in ai industry. He  --predict next letter is-- o
in ai industry. He o --predict next letter is-- b
